# Linear Attention Practice

**Topic:** Linear Attention &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `Q`, `K`, `V`: arrays of shape `(2, 50, 16)`.
- Feature map: `phi(x) = elu(x) + 1`, where `elu(x) = x` for `x > 0` and `exp(x) − 1` otherwise.
- Non-causal (full) attention for this exercise.

## Your task

With **NumPy only**, compute linear attention *the O(n) way*:

1. `S = phi(K)ᵀ @ V` (shape `(16, 16)` per batch) and `z = phi(K).sum(axis=seq)`.
2. `out_i = (phi(q_i) @ S) / (phi(q_i) @ z)`, vectorized over positions.
3. Set `lin_out` to the `(2, 50, 16)` result.

Note the order: you must **never** build a 50×50 attention matrix.

## Evaluation

Max absolute difference from the reference (computed in the evaluation cell with the same feature map) must be below 1e-8.


In [ ]:
import numpy as np

rng = np.random.default_rng(451)
B, N, D = 2, 50, 16
Q = rng.normal(0, 1.0, (B, N, D))
K = rng.normal(0, 1.0, (B, N, D))
V = rng.normal(0, 1.0, (B, N, D))
print(f"Q/K/V {Q.shape}: produce lin_out of the same shape, in O(n)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: reference computed with the same feature map (same math)
# =====================================================================
import numpy as np

if "lin_out" not in dir():
    raise NameError("Define `lin_out` in the solution cell before running this cell.")

def _phi(t):
    return np.where(t > 0, t, np.exp(t) - 1.0) + 1.0

_qp, _kp = _phi(Q), _phi(K)
_num = _qp @ (_kp.transpose(0, 2, 1) @ V)               # same math, written in one line
_den = np.einsum("bnd,bd->bn", _qp, _kp.sum(axis=1))
_ref = _num / _den[..., None]

_out = np.asarray(lin_out, dtype=float)
if _out.shape != _ref.shape:
    raise ValueError(f"`lin_out` has shape {_out.shape}, expected {_ref.shape}.")
_err = float(np.max(np.abs(_out - _ref)))
print("=" * 52)
print(f"Max absolute error vs reference: {_err:.2e}   (need < 1e-8)")
print("=" * 52)
print("PASSED: attention in linear time!" if _err < 1e-8 else "NOT PASSED yet: check phi and the normalizer.")


## Solution

Apply the `elu(x)+1` feature map, then use associativity: precompute `S = phi(K)T V` (16x16) and `z = phi(K).sum(seq)` once, so each query costs O(d^2) instead of O(n·d), with no 50x50 matrix anywhere.

```python
import numpy as np

def phi(t):
    return np.where(t > 0, t, np.exp(t) - 1.0) + 1.0

qp, kp = phi(Q), phi(K)

S = kp.transpose(0, 2, 1) @ V            # (B, 16, 16): shared across positions
z = kp.sum(axis=1)                       # (B, 16)

num = qp @ S                             # (B, 50, 16)
den = np.einsum("bnd,bd->bn", qp, z)     # (B, 50)
lin_out = num / den[..., None]
print(lin_out.shape)
```
